# EEE429 Final — MNIST CNN Quantization & HLS Export

3개 양자화 후보를 학습하고 `artifacts/hls/`에 HW용 산출물을 뽑는 노트북.

| Section | Format | Checkpoint dir | HLS output |
|---------|--------|----------------|------------|
| **0. Setup** | — | — | — |
| **1. Shared** | test vectors | — | `artifacts/hls/shared/` |
| **2. Fixed 16/7** | ap_fixed⟨16,7⟩ | `milestones/` | `artifacts/hls/fixed16/weights.h` |
| **3. Fixed 8/4** | ap_fixed⟨8,4⟩ | `milestones_fixed8/` | `artifacts/hls/fixed8/weights.h` |
| **4. BNN** | ±1 + BN + FC | `milestones_bnn/` | `artifacts/hls/bnn/weights.h` |
| **5. Export** | CLI wrapper | — | `python hls_export.py all` |

**실행 순서:** Setup → (학습할 Section만) → Section 5 Export

**전처리 (모든 variant 공통):** `ToTensor` + `Normalize(0.5, 0.5)` → [-1, +1]


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from dataclasses import dataclass, InitVar, field, asdict

### DATASET and PARAMETER

@dataclass(frozen=True)
class HyperParameter:
    batch_size: int
    num_epochs: int
    learning_rate: float

@dataclass
class MNIST:
    root: InitVar[str]
    param: InitVar[HyperParameter]

    train: DataLoader = field(init=False)
    test:  DataLoader = field(init=False)
    total: DataLoader = field(init=False)

    def __post_init__(self, root, param):
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        train_set = datasets.MNIST(
            root=root,
            train=True,
            download=True,
            transform=transform,
        )
        test_set = datasets.MNIST(
            root=root,
            train=False,
            download=True,
            transform=transform,
        )
        self.train = DataLoader(
            dataset=train_set,
            batch_size=param.batch_size, shuffle=True
        )
        self.test = DataLoader(
            dataset=test_set,
            batch_size=param.batch_size, shuffle=False
        )
        self.total = DataLoader(
            dataset=train_set+test_set,
            batch_size=param.batch_size, shuffle=False
        )

import os
from pathlib import Path
from datetime import datetime

### TRAIN / EVALUATION ROUTINE

@dataclass
class Routine:
    device: object
    criterion: object
    optimizer: object

    def _routine(self, model, loader):
        num_hit, running_loss = 0, 0.0

        for images, labels in loader:
            images = images.to(self.device)
            labels = labels.to(self.device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            loss = self.criterion(outputs, labels)

            if torch.is_grad_enabled():
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

            num_hit += (predicted==labels).sum().item()
            running_loss += loss.item()

        return num_hit / len(loader.dataset), \
               running_loss / len(loader)

    def train(self, model, loader):
        model.train()
        return self._routine(model, loader)

    def test(self, model, loader):
        model.eval()
        with torch.no_grad():
            return self._routine(model, loader)

@dataclass
class Milestone:
    parent: str | Path
    model_name: str
    ext: str
    now: str = field(init=False)

    def __post_init__(self):
        self.now = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
        self.parent = Path(self.parent)

        if not self.parent.exists():
            self.parent.mkdir()

    def __iter__(self):
        files = self.parent.glob(f"{self.model_name}-*.{self.ext}")
        return iter(sorted(files, key=lambda f: f.name, reverse=True))

    def save(self, **kwargs):
        torch.save(kwargs, self.parent / f"{self.model_name}-{self.now}.{self.ext}")

# device (cuda → mps → cpu)
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('device:', device)


---
## Section 1 — Shared test vectors

코시뮬/bring-up용 MNIST 이미지·라벨. **Export 전에 한 번 실행.**


In [ ]:
from hls_export import export_shared

export_shared()  # → artifacts/hls/shared/images.h, MNIST_DATASET_*.npy


---
## Section 2 — Fixed 16/7 (ap_fixed⟨16,7⟩)

Graph: `Input→Q→Conv→Q→ReLU` (×3) → Pool → Q → FC  
Range ≈ [-64, 64), step 2⁻⁹


In [ ]:
### FIXED-POINT EMULATION: ap_fixed<16,7>
FIXED_W = 16
FIXED_I = 7
FIXED_F = FIXED_W - FIXED_I

FIXED_SCALE = 2 ** FIXED_F
FIXED_MIN = -(2 ** (FIXED_I - 1))
FIXED_MAX = (2 ** (FIXED_I - 1)) - (1 / FIXED_SCALE)

def fixed_point_round(x):
    """
    Emulate signed ap_fixed<16 ,7>
    W = 16
    I = 7
    F = 9
    Range = [-64, 64)
    Step = 2^-9
    """
    x = torch.clamp(x, FIXED_MIN, FIXED_MAX)
    x = torch.round(x * FIXED_SCALE) / FIXED_SCALE
    return x

class FixedPointQuantize(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x):
        return fixed_point_round(x)

    @staticmethod
    def backward(ctx, grad_output):
        # STE
        return grad_output

### QUANTIZED CONVOLUTION

class QuantizedConv2d(nn.Conv2d):

    def forward(self, x):

        q_weight = FixedPointQuantize.apply(self.weight)

        if self.bias is not None:
          q_bias = FixedPointQuantize.apply(self.bias)
        else:
          q_bias = None
        return F.conv2d(
            x,
            q_weight,
            q_bias,
            self.stride,
            self.padding,
            self.dilation,
            self.groups
        )


### QUANTIZED LINEAR

class QuantizedLinear(nn.Linear):

    def forward(self, x):

        q_weight = FixedPointQuantize.apply(self.weight)

        if self.bias is not None:
          q_bias = FixedPointQuantize.apply(self.bias)
        else:
          q_bias = None

        return F.linear(x, q_weight, q_bias)


### QUANTIZED CNN DEFINITION

class QUANTIZED_MNISTCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = QuantizedConv2d(1, 16, 3, 1, bias=True)    # 2D Convolution1 : [in=(1,28,28), kernel=(3,3)]  >>>   2D Convolution2 [in=(16,26,26), kernel=(3,3)] 이므로  output channel =16, 28>26 이므로 stride 1
        self.conv2 = QuantizedConv2d(16, 32, 3, 1, bias=True)   # 2D Convolution2 : [in=(16,26,26), kernel=(3,3)] >>>   2D Convolution3 [in=(32,24,24), kernel=(3,3)] 이므로  output channel =32, 26>24 이므로 stride 1
        self.conv3 = QuantizedConv2d(32, 32, 3, 1, bias=True)   # 2D Convolution3 [in=(32,24,24), kernel=(3,3)]   >>>   max pooling [kernel=(2,2)] 로부터 3872 = 32*11*11, max pooling 전은 32*22*22 로 output channel =32, 24>22 이므로 stride 1
        self.pool = nn.MaxPool2d(2)
        self.fc = QuantizedLinear(3872, 10, bias=True)

    def forward(self, x):

        # input quantization
        x = FixedPointQuantize.apply(x)

        x = self.conv1(x)
        x = FixedPointQuantize.apply(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = FixedPointQuantize.apply(x)
        x = F.relu(x)

        x = self.conv3(x)
        x = FixedPointQuantize.apply(x)
        x = F.relu(x)

        x = self.pool(x)

        x = FixedPointQuantize.apply(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x


In [ ]:
### Fixed 16/7 — Train
param = HyperParameter(batch_size=256, num_epochs=5, learning_rate=1e-3)
mnist = MNIST('./data', param)
model = QUANTIZED_MNISTCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=param.learning_rate)
routine = Routine(device, criterion, optimizer)

milestone = Milestone("./milestones", type(model).__name__,'pth')
print(f"Train Starts at: {milestone.now}")
print(f"Target Model: {milestone.model_name}")
print('')
print(f"| {'EPOCH': ^5} | {'TRAIN': ^33} | {'TEST': ^33} |")
print(f"| {'     ': ^5} "
      f"| {'ACCURACY': ^15} | {'LOSS': ^15} "
      f"| {'ACCURACY': ^15} | {'LOSS': ^15} |")

checkpoint = None
for epoch in range(param.num_epochs):
    train_acc, train_loss = routine.train(model, mnist.train)
    test_acc, test_loss = routine.test(model, mnist.test)

    print(f"| {epoch: ^5} "
          f"| {train_acc: ^15.2%} | {train_loss: ^15.5} "
          f"| {test_acc: ^15.2%} | {test_loss: ^15.5} |", end='')

    if (checkpoint is None) or (checkpoint['accuracy'] < test_acc):
        checkpoint = dict(
            epoch=epoch,
            accuracy=test_acc,
            param=asdict(param),
            model=model.state_dict(),
            optim=optimizer.state_dict(),
        )
        milestone.save(**checkpoint)
        print(' *', end='')
    print('')


In [ ]:
model_name = 'QUANTIZED_MNISTCNN'
test_ms = Milestone(milestone.parent, model_name, milestone.ext)
test_model = globals()[model_name]()
test_model.eval()

with torch.no_grad():
    for ms_file in test_ms:
        checkpoint = torch.load(ms_file)

        test_model.load_state_dict(checkpoint['model'])
        test_model.to(device)
        accuracy, _ = routine.test(test_model, mnist.total)
        print(ms_file.stem, ':\t', f"{accuracy: .2%}")


---
## Section 3 — Fixed 8/4 (ap_fixed⟨8,4⟩)

16-bit와 동일 그래프. Range ≈ [-8, 8), step 2⁻⁴


In [ ]:
#_____________________________________FIXED 8/4 (ap_fixed<8,4>)________________________________________
# Same graph as 16-bit; range ≈ [-8, 8), step 2^-4

FIXED8_W = 8
FIXED8_I = 4
FIXED8_F = FIXED8_W - FIXED8_I
FIXED8_SCALE = 2 ** FIXED8_F
FIXED8_MIN = -(2 ** (FIXED8_I - 1))
FIXED8_MAX = (2 ** (FIXED8_I - 1)) - (1 / FIXED8_SCALE)

def fixed8_point_round(x):
    x = torch.clamp(x, FIXED8_MIN, FIXED8_MAX)
    return torch.round(x * FIXED8_SCALE) / FIXED8_SCALE

class Fixed8PointQuantize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return fixed8_point_round(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output

class Quantized8Conv2d(nn.Conv2d):
    def forward(self, x):
        qw = Fixed8PointQuantize.apply(self.weight)
        qb = Fixed8PointQuantize.apply(self.bias) if self.bias is not None else None
        return F.conv2d(x, qw, qb, self.stride, self.padding, self.dilation, self.groups)

class Quantized8Linear(nn.Linear):
    def forward(self, x):
        qw = Fixed8PointQuantize.apply(self.weight)
        qb = Fixed8PointQuantize.apply(self.bias) if self.bias is not None else None
        return F.linear(x, qw, qb)

class QUANTIZED8_MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = Quantized8Conv2d(1, 16, 3, 1, bias=True)
        self.conv2 = Quantized8Conv2d(16, 32, 3, 1, bias=True)
        self.conv3 = Quantized8Conv2d(32, 32, 3, 1, bias=True)
        self.pool = nn.MaxPool2d(2)
        self.fc = Quantized8Linear(3872, 10, bias=True)

    def forward(self, x):
        x = Fixed8PointQuantize.apply(x)
        x = self.conv1(x)
        x = Fixed8PointQuantize.apply(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = Fixed8PointQuantize.apply(x)
        x = F.relu(x)
        x = self.conv3(x)
        x = Fixed8PointQuantize.apply(x)
        x = F.relu(x)
        x = self.pool(x)
        x = Fixed8PointQuantize.apply(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


In [ ]:
### Fixed 8/4 Training
param8 = HyperParameter(
    batch_size=256,
    num_epochs=10,
    learning_rate=1e-3,
)
mnist8 = MNIST('./data', param8)
model8 = QUANTIZED8_MNISTCNN().to(device)
criterion8 = nn.CrossEntropyLoss()
optimizer8 = optim.Adam(model8.parameters(), lr=param8.learning_rate)
routine8 = Routine(device, criterion8, optimizer8)

milestone8 = Milestone("./milestones_fixed8", type(model8).__name__, 'pth')
print(f"Train Starts at: {milestone8.now}")
print(f"Target Model: {milestone8.model_name}")
print('')
print(f"| {'EPOCH': ^5} | {'TRAIN': ^33} | {'TEST': ^33} |")
print(f"| {'     ': ^5} "
      f"| {'ACCURACY': ^15} | {'LOSS': ^15} "
      f"| {'ACCURACY': ^15} | {'LOSS': ^15} |")

checkpoint8 = None
for epoch in range(param8.num_epochs):
    train_acc, train_loss = routine8.train(model8, mnist8.train)
    test_acc, test_loss = routine8.test(model8, mnist8.test)
    print(f"| {epoch: ^5} "
          f"| {train_acc: ^15.2%} | {train_loss: ^15.5} "
          f"| {test_acc: ^15.2%} | {test_loss: ^15.5} |", end='')
    if (checkpoint8 is None) or (checkpoint8['accuracy'] < test_acc):
        checkpoint8 = dict(
            epoch=epoch,
            accuracy=test_acc,
            param=asdict(param8),
            model=model8.state_dict(),
            optim=optimizer8.state_dict(),
        )
        milestone8.save(**checkpoint8)
        print(' *', end='')
    print('')


In [ ]:
### Fixed 8/4 Evaluation — best checkpoint on full 70k dataset
model_name8 = 'QUANTIZED8_MNISTCNN'
test_ms8 = Milestone('./milestones_fixed8', model_name8, 'pth')
test_model8 = globals()[model_name8]()
test_model8.eval()

with torch.no_grad():
    for ms_file in test_ms8:
        ckpt8 = torch.load(ms_file, map_location='cpu', weights_only=False)
        test_model8.load_state_dict(ckpt8['model'])
        test_model8.to(device)
        accuracy8, _ = routine8.test(test_model8, mnist8.total)
        print(ms_file.stem, ':\t', f"{accuracy8: .2%}")

### Fixed 8/4 Export → artifacts/hls/fixed8/weights.h


---
## Section 4 — BNN (Binary Neural Network)

Graph: `Normalize → sign(±1) → Conv(±1 w) → BN → sign → … → Pool → FC(float)`  
eval 전 `calibrate_bn()` 필수


In [ ]:
#_____________________________________BNN (Binary Neural Network)________________________________________
# weights & activations → {-1, +1}
# BatchNorm before each binarization (standard BNN)
# Clipped STE: gradient passes only where |x| <= 1

def binary_sign(x: torch.Tensor) -> torch.Tensor:
    """Binarize to {-1, +1}. Values >= 0 map to +1."""
    return (x >= 0).float() * 2 - 1

class BinaryQuantize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return binary_sign(x)

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        # Clipped STE: pass gradient only where |x| <= 1
        return grad_output * (x.abs() <= 1).float()

class BinaryConv2d(nn.Conv2d):
    def forward(self, x):
        w_b = BinaryQuantize.apply(self.weight)
        return self._conv_forward(x, w_b, self.bias)

### BINARY MNISTCNN
class BINARY_MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # bias=False: BatchNorm absorbs bias
        self.conv1 = BinaryConv2d(1,  16, 3, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(16)
        self.conv2 = BinaryConv2d(16, 32, 3, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(32)
        self.conv3 = BinaryConv2d(32, 32, 3, 1, bias=False)
        self.bn3   = nn.BatchNorm2d(32)
        self.pool  = nn.MaxPool2d(2)
        self.fc    = nn.Linear(3872, 10, bias=True)  # real-valued output layer

    def forward(self, x):
        x = BinaryQuantize.apply(x)          # input → {-1, +1}

        x = self.conv1(x)
        x = self.bn1(x)
        x = BinaryQuantize.apply(x)          # BN → {-1, +1}

        x = self.conv2(x)
        x = self.bn2(x)
        x = BinaryQuantize.apply(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = BinaryQuantize.apply(x)

        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


def calibrate_bn(model, loader, device, passes=1):
    """BNN: eval() 전 running stats 갱신 — test acc 튐 방지"""
    was_training = model.training
    model.train()
    with torch.no_grad():
        for _ in range(passes):
            for images, _ in loader:
                model(images.to(device))
    model.train(was_training)


In [ ]:
### BNN Training
# BNN converges slower than QAT → more epochs, lower lr schedule
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

param_bnn = HyperParameter(
    batch_size=256,
    num_epochs=20,
    learning_rate=1e-3,
)
mnist_bnn   = MNIST('./data', param_bnn)
model_bnn   = BINARY_MNISTCNN().to(device)
crit_bnn    = nn.CrossEntropyLoss()
optim_bnn   = optim.Adam(model_bnn.parameters(), lr=param_bnn.learning_rate)
routine_bnn = Routine(device, crit_bnn, optim_bnn)

milestone_bnn = Milestone("./milestones_bnn", type(model_bnn).__name__, 'pth')
print(f"Train Starts at: {milestone_bnn.now}")
print(f"Target Model   : {milestone_bnn.model_name}")
print(f"Device         : {device}")
print('')
print(f"| {'EPOCH':^5} | {'TRAIN':^33} | {'TEST':^33} |")
print(f"| {'':^5} | {'ACCURACY':^15} | {'LOSS':^15} | {'ACCURACY':^15} | {'LOSS':^15} |")

ckpt_bnn = None
for epoch in range(param_bnn.num_epochs):
    tr_acc, tr_loss = routine_bnn.train(model_bnn, mnist_bnn.train)
    calibrate_bn(model_bnn, mnist_bnn.train, device)  # BN running stats 갱신 후 eval
    te_acc, te_loss = routine_bnn.test(model_bnn,  mnist_bnn.test)

    saved = ''
    if ckpt_bnn is None or ckpt_bnn['accuracy'] < te_acc:
        ckpt_bnn = dict(
            epoch=epoch, accuracy=te_acc,
            param=asdict(param_bnn),
            model=model_bnn.state_dict(),
            optim=optim_bnn.state_dict(),
        )
        milestone_bnn.save(**ckpt_bnn)
        saved = ' *'

    print(f"| {epoch:^5} | {tr_acc:^15.2%} | {tr_loss:^15.5} "
          f"| {te_acc:^15.2%} | {te_loss:^15.5} |{saved}")


In [ ]:
### BNN Evaluation — best checkpoint on full 70k dataset
model_name_bnn = 'BINARY_MNISTCNN'
test_ms_bnn    = Milestone(milestone_bnn.parent, model_name_bnn, milestone_bnn.ext)
eval_model_bnn = BINARY_MNISTCNN()
eval_model_bnn.eval()

with torch.no_grad():
    for ms_file in test_ms_bnn:
        ck = torch.load(ms_file, weights_only=False)
        eval_model_bnn.load_state_dict(ck['model'])
        eval_model_bnn.to(device)
        calibrate_bn(eval_model_bnn, mnist_bnn.train, device)
        acc, _ = routine_bnn.test(eval_model_bnn, mnist_bnn.total)
        print(ms_file.stem, ':\t', f'{acc: .2%}')


---
## Section 5 — HLS Export (weights.h + npy)

학습된 checkpoint에서 HW용 헤더 생성. 터미널에서도 가능:
```bash
python hls_export.py shared
python hls_export.py fixed16
python hls_export.py fixed8
python hls_export.py bnn
python hls_export.py all
```


In [ ]:
from hls_export import export_shared, export_variant, export_all_trained

# 필요한 줄만 주석 해제 후 실행
# export_shared()
# export_variant('fixed16')   # milestones/ QUANTIZED_MNISTCNN-*.pth
# export_variant('fixed8')    # milestones_fixed8/ (Section 3 학습 후)
# export_variant('bnn')       # milestones_bnn/
export_all_trained()          # 학습된 variant 전부 (없으면 skip)
